# Demo - Catastrophic Forgetting
**Day 3 - Session 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day3/demos/notebooks/demo-catastrophic-forgetting.ipynb)

**Goal:** Show that training a model hard on a new task can erase what it
learned on an earlier task.

We build one small neural net and two tasks over the same 2D input space. Task
A and Task B use different decision boundaries. We train on Task A first and
measure its accuracy, then train only on Task B and measure both again. Task B
accuracy climbs while Task A accuracy collapses: the weights that solved A were
overwritten.

No API key, no downloads. Pure PyTorch on CPU.

In [1]:
# Scoped install for this demo (safe to re-run; torch is preinstalled on Colab).
%pip install -q torch

In [2]:
import torch
import torch.nn as nn

torch.manual_seed(0)

## 1. Helper functions
`make_task` builds a linearly separable task from a decision boundary,
`accuracy` scores predictions, and `train` runs a short optimization loop.

In [3]:
def make_task(boundary):
    """Return (X, y): label 1 when x0 + x1 > boundary, else 0."""
    x = torch.rand(400, 2) * 2 - 1  # points in [-1, 1] x [-1, 1]
    y = (x[:, 0] + x[:, 1] > boundary).long()
    return x, y


def accuracy(model, x, y):
    """Fraction of correct predictions."""
    with torch.no_grad():
        preds = model(x).argmax(dim=1)
    return (preds == y).float().mean().item()


def train(model, x, y, steps=300):
    """Train the model to convergence on one task."""
    opt = torch.optim.Adam(model.parameters(), lr=0.05)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(steps):
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()

## 2. Train on Task A only
Build two tasks with different boundaries and one small network, then train it
on Task A and measure Task A accuracy.

In [4]:
task_a = make_task(boundary=0.5)
task_b = make_task(boundary=-0.5)

model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))

train(model, *task_a)
a_after_a = accuracy(model, *task_a)
print("After training on Task A only:")
print(f"  Task A accuracy: {a_after_a:.2f}")

After training on Task A only:
  Task A accuracy: 1.00


## 3. Then train on Task B only
Now train the same weights on Task B and re-measure both tasks. Watch Task A
accuracy drop as Task B accuracy rises.

In [5]:
train(model, *task_b)
a_after_b = accuracy(model, *task_a)
b_after_b = accuracy(model, *task_b)
print("After then training on Task B only:")
print(f"  Task A accuracy: {a_after_b:.2f}  (dropped from {a_after_a:.2f})")
print(f"  Task B accuracy: {b_after_b:.2f}")

After then training on Task B only:
  Task A accuracy: 0.56  (dropped from 1.00)
  Task B accuracy: 1.00


## Expected output
- After training on Task A only, Task A accuracy is high (near 1.00).
- After training on Task B only, Task B accuracy is high but Task A accuracy
  drops sharply: the weights that solved A were overwritten.

**Takeaway:** narrow, hard fine-tuning overwrites earlier skills. Lower the
learning rate, use fewer epochs, or prefer PEFT to reduce it.